# 01b — Contrôle Qualité & Alignement (Si 520 cm⁻¹)

In [ ]:

import os, sys, numpy as np, matplotlib.pyplot as plt
sys.path.append(os.path.join("..","src"))
from raman_ai.io.standards import load_reference_spectra_mode

def qc_basic(wavenumber, intensity):
    wn = np.asarray(wavenumber, float)
    y = np.asarray(intensity, float)
    flags = {}
    flags["has_nan"] = bool(np.isnan(wn).any() or np.isnan(y).any())
    flags["has_inf"] = bool(np.isinf(wn).any() or np.isinf(y).any())
    flags["len_ok"] = (len(wn) >= 5 and len(wn) == len(y))
    if len(y) > 5:
        hf = y - np.convolve(y, np.ones(5)/5.0, mode="same")
        snr = (np.max(y) - np.min(y)) / (np.std(hf) + 1e-12)
    else:
        snr = np.nan
    flags["snr_proxy"] = float(snr)
    flags["wn_monotone"] = bool(np.all(np.diff(wn) >= 0))
    return flags

def align_to_si520(wavenumber, intensity, window=(500,540), target=520.0):
    wn = np.asarray(wavenumber, float)
    y = np.asarray(intensity, float)
    mask = (wn >= window[0]) & (wn <= window[1])
    if mask.sum() < 1:
        return wn, y, None
    idx = np.argmax(y[mask])
    peak_wn = wn[mask][idx]
    shift = target - peak_wn
    return wn + shift, y, float(shift)

root = os.path.join("..","data","standards")
specs, metas = load_reference_spectra_mode(root, mode="synthetic")

qc = [ (s.name, qc_basic(s.wavenumber, s.intensity)) for s in specs ]
qc[:3]


In [ ]:

# Visualiser un alignement
s = specs[0]
wn_al, y, shift = align_to_si520(s.wavenumber, s.intensity)
plt.figure(); plt.plot(s.wavenumber, s.intensity); plt.plot(wn_al, y); plt.title(f"Align shift={shift} cm^-1"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()
